## RNN Model Building LSTM (Long Short-Term Memory)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
import edf

original_df, _, _, _, _ = edf.get_data_for_modeling()

In [ ]:
# Prepare the data for LSTM model
demand_data = original_df[['Demand']].values
training_data_len = int(np.ceil(len(demand_data) * 0.8))

# Preprocessing: Standardize the data
scaler = StandardScaler()
scaled_data = scaler.fit_transform(demand_data)

train_data = scaled_data[0:training_data_len, :]

X_train = []
Y_train = []

# Create a sliding window of 24 hours to predict the next hour's demand
for i in range(24, len(train_data)):
    X_train.append(train_data[i-24:i, 0])
    Y_train.append(train_data[i, 0])

X_train, Y_train = np.array(X_train), np.array(Y_train)
X_train = np.reshape(X_train, (X_train.shape[0], X_train.shape[1], 1))

In [ ]:
# Building the LSTM model
model = keras.models.Sequential([
    keras.layers.LSTM(64, return_sequences=True, input_shape=(X_train.shape[1], 1)),
    keras.layers.LSTM(64, return_sequences=False),
    keras.layers.Dense(128, activation='relu'),
    keras.layers.Dense(64, activation='relu'),
    keras.layers.Dense(1)
])

model.summary()
model.compile(optimizer='adam', loss='mae', metrics=[keras.metrics.RootMeanSquaredError()])

history = model.fit(X_train, Y_train, epochs=10, batch_size=32, validation_split=0.1)

In [ ]:
# Prepare the test data
test_data = scaled_data[training_data_len - 24:, :]
X_test = []
Y_actual = demand_data[training_data_len:, :]

for i in range(24, len(test_data)):
    X_test.append(test_data[i-24:i, 0])

X_test = np.array(X_test)
X_test = np.reshape(X_test, (X_test.shape[0], X_test.shape[1], 1))

# Make predictions
predictions = model.predict(X_test)
predictions = scaler.inverse_transform(predictions)

# Evaluating the LSTM model
mae_lstm = mean_absolute_error(Y_actual, predictions)
rmse_lstm = np.sqrt(mean_squared_error(Y_actual, predictions))
print('LSTM RMSE:', rmse_lstm)
print('LSTM MAE:', mae_lstm)

In [ ]:
# Visualize the predictions vs actual values
plt.figure(figsize=(15,6))
plt.plot(range(len(Y_actual)), Y_actual, label='Actual Demand', color='blue')
plt.plot(range(len(predictions)), predictions, label='Predicted Demand', color='Red', linestyle='--')
plt.title("LSTM Electricity Demand Forecast")
plt.xlabel("Time Steps")
plt.ylabel("Electricity Demand")
plt.legend()
plt.show()